In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load CSV files into memory and store in kernel variables
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Ensure timestamp is integer and create timezone-aware datetime in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        # Convert to datetime with UTC then convert to Asia/Shanghai
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) For each file, return the total row count.
counts = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv'],
    'rows': [len(metric_df), len(trace_df), len(log_df), len(error_logs_df)]
})

# 2) Up to 20 distinct cmdb_id values across all files (union) sorted alphabetically.
cmdb_union = pd.Series(
    sorted(
        set(metric_df.get('cmdb_id', pd.Series(dtype=str)).dropna().unique())
        | set(trace_df.get('cmdb_id', pd.Series(dtype=str)).dropna().unique())
        | set(log_df.get('cmdb_id', pd.Series(dtype=str)).dropna().unique())
        | set(error_logs_df.get('cmdb_id', pd.Series(dtype=str)).dropna().unique())
    )
).head(20).to_frame(name='cmdb_id')

# 3) metric.csv: group (cmdb_id, kpi_name) -> top 20 by count with P95 and P5 (global percentiles)
if not metric_df.empty:
    # Ensure value is numeric
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    metric_group = metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    metric_summary = metric_group.agg(
        count='count',
        p95=lambda x: x.quantile(0.95),
        p05=lambda x: x.quantile(0.05)
    ).reset_index().sort_values('count', ascending=False).head(20)
    # Round percentiles for compactness
    metric_summary['p95'] = metric_summary['p95'].round(6)
    metric_summary['p05'] = metric_summary['p05'].round(6)
else:
    metric_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','count','p95','p05'])

# 4) trace.csv: group (cmdb_id, trace_name) -> top 20 by count with P95
if not trace_df.empty:
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
    trace_group = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    trace_summary = trace_group.agg(
        count='count',
        p95=lambda x: x.quantile(0.95)
    ).reset_index().sort_values('count', ascending=False).head(20)
    trace_summary['p95'] = trace_summary['p95'].round(6)
else:
    trace_summary = pd.DataFrame(columns=['cmdb_id','trace_name','count','p95'])

# 5) log.csv: group (cmdb_id, log_name) -> top 20 by count with P95
if not log_df.empty:
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
    log_group = log_df.groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    log_summary = log_group.agg(
        count='count',
        p95=lambda x: x.quantile(0.95)
    ).reset_index().sort_values('count', ascending=False).head(20)
    log_summary['p95'] = log_summary['p95'].round(6)
else:
    log_summary = pd.DataFrame(columns=['cmdb_id','log_name','count','p95'])

# 6) error_logs.csv: up to 10 cmdb_id entries with count, earliest timestamp, latest timestamp, up to 3 example messages truncated to 200 chars
if not error_logs_df.empty:
    # Ensure messages are strings
    error_logs_df['message'] = error_logs_df['message'].astype(str)
    def collect_examples(msgs):
        # take up to 3 distinct messages
        uniq = pd.Series(msgs.dropna().unique())
        truncated = uniq.apply(lambda s: (s[:200]) if isinstance(s, str) else s)
        return truncated.head(3).tolist()
    err_grp = error_logs_df.groupby('cmdb_id')
    error_summary = err_grp.agg(
        count=('message', 'count'),
        earliest=('timestamp', 'min'),
        latest=('timestamp', 'max'),
        messages=('message', lambda x: collect_examples(x))
    ).reset_index().sort_values('count', ascending=False).head(10)
    # Format earliest/latest as ISO strings in timezone Asia/Shanghai (already tz-aware)
    error_summary['earliest'] = error_summary['earliest'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_summary['latest'] = error_summary['latest'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    error_summary = pd.DataFrame(columns=['cmdb_id','count','earliest','latest','messages'])

# Keep outputs compact: show counts, cmdb_union, and the group summaries (trimmed to required sizes)
counts, cmdb_union, metric_summary, trace_summary, log_summary, error_summary

```
Out[1]:
```
import pandas as pd

# Build a concise plain-English summary from previously loaded results (counts, cmdb_union, metric_summary, trace_summary, log_summary, error_summary)

rows_map = {r['file']: int(r['rows']) for _, r in counts.iterrows()}
cmdb_list = cmdb_union['cmdb_id'].tolist()

def fmt_rows(df, cols, n=5):
    out = []
    for _, r in df[cols].head(n).iterrows():
        parts = [f"{c}={r[c]}" for c in cols if c in r.index]
        out.append(", ".join(parts))
    return out

metric_lines = fmt_rows(metric_summary, ['cmdb_id', 'kpi_name', 'count', 'p95', 'p05'], n=5)
trace_lines = fmt_rows(trace_summary, ['cmdb_id', 'trace_name', 'count', 'p95'], n=5)
log_lines = fmt_rows(log_summary, ['cmdb_id', 'log_name', 'count', 'p95'], n=5)

summary = (
    "Summary of loaded telemetry files:\n\n"
    f"1) Row counts:\n"
    f"   - metric.csv: {rows_map.get('metric.csv', 'N/A')} rows\n"
    f"   - trace.csv: {rows_map.get('trace.csv', 'N/A')} rows\n"
    f"   - log.csv: {rows_map.get('log.csv', 'N/A')} rows\n"
    f"   - error_logs.csv: {rows_map.get('error_logs.csv', 'N/A')} rows\n\n"
    f"2) Up to 20 distinct cmdb_id values (alphabetical):\n"
    f"   {', '.join(cmdb_list)}\n\n"
    "3) Top metric (cmdb_id, kpi_name) pairs by row count (showing top 5):\n"
    + "\n".join([f"   - {l}" for l in metric_lines]) + "\n\n"
    "4) Top trace (cmdb_id, trace_name) pairs by row count (showing top 5):\n"
    + "\n".join([f"   - {l}" for l in trace_lines]) + "\n\n"
    "5) Top log (cmdb_id, log_name) pairs by row count (showing top 5):\n"
    + "\n".join([f"   - {l}" for l in log_lines]) + "\n\n"
    "6) error_logs.csv summary:\n"
    "   - No error log rows found (empty file).\n\n"
    "Notes:\n"
    " - Percentiles shown are global (computed on all rows for each pair).\n    p95 = 95th percentile, p05 = 5th percentile where provided.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows
0      metric.csv  27285
1       trace.csv   6960
2         log.csv    478
3  error_logs.csv      0,           cmdb_id
0            IG01
1            IG02
2            MG01
3            MG02
4         Mysql01
5         Mysql02
6         Redis01
7         Redis02
8    ServiceTest1
9   ServiceTest10
10  ServiceTest11
11   ServiceTest2
12   ServiceTest3
13   ServiceTest4
14   ServiceTest5
15   ServiceTest6
16   ServiceTest7
17   ServiceTest8
18   ServiceTest9
19       Tomcat01,            cmdb_id                                           kpi_name  count           p95           p05
1560      apache02                            OSLinux-CPU_CPU_CPULoad     31  9.500000e-02  0.000000e+00
6             IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.500000e+01  6.700000e+01
1624      apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  7.000000e+00  7.000000e+00
1630      apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.214374e+06  6.212748e+06
1546      apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00  0.000000e+00
1564      apache02                        OSLinux-CPU_CPU_CPUidleutil     31  9.950564e+01  9.899850e+01
3             IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.800194e+07  5.798357e+07
1619      apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00  0.000000e+00
2             IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  8.703499e+08  1.770111e+08
1563      apache02                             OSLinux-CPU_CPU_CPUWio     31  3.179000e-01  1.246300e-02
1623      apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  4.000000e-01  4.000000e-01
0             IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09
1071  ServiceTest7                                             app.sr     31  1.000000e+02  1.000000e+02
1056  ServiceTest4                                            app.cnt     31  2.745000e+02  4.550000e+01
1057  ServiceTest4                                            app.mrt     31  3.849125e+00  2.117815e+00
1058  ServiceTest4                                             app.rr     31  1.000000e+02  9.851500e+01
1075  ServiceTest8                                             app.sr     31  1.000000e+02  1.000000e+02
1076  ServiceTest9                                            app.cnt     31  4.320000e+02  1.115000e+02
1077  ServiceTest9                                            app.mrt     31  4.161650e-01  1.911700e-01
1078  ServiceTest9                                             app.rr     31  1.000000e+02  1.000000e+02,       cmdb_id                     trace_name  count          p95
263  dockerB2        trace.to_MG02.row_count     30   316.100000
262  dockerB2     trace.to_MG02.duration_p95     30     1.019542
261  dockerB2    trace.to_MG02.duration_mean     30     0.135183
260  dockerB2        trace.to_MG01.row_count     30   304.150000
3        IG01       trace.self.duration_mean     30     0.862223
221  dockerA2           trace.self.row_count     30  3263.600000
222  dockerA2    trace.to_MG01.duration_mean     30     0.157075
223  dockerA2     trace.to_MG01.duration_p95     30     1.018280
224  dockerA2        trace.to_MG01.row_count     30    74.400000
225  dockerA2    trace.to_MG02.duration_mean     30     0.173089
226  dockerA2     trace.to_MG02.duration_p95     30     1.018185
227  dockerA2        trace.to_MG02.row_count     30    93.800000
228  dockerB1  trace.from_MG01.duration_mean     30     1.366040
213  dockerA2  trace.from_MG02.duration_mean     30     0.817424
214  dockerA2   trace.from_MG02.duration_p95     30     2.237890
215  dockerA2      trace.from_MG02.row_count     30   231.000000
219  dockerA2       trace.self.duration_mean     30     0.108773
220  dockerA2        trace.self.duration_p95     30     1.017000
205  dockerA1     trace.to_MG01.duration_p95     30     1.015465
206  dockerA1        trace.to_MG01.row_count     30    77.300000,      cmdb_id          log_name  count        p95
2   Tomcat01   log.error_count     31     0.0000
7   Tomcat02     log.row_count     31   610.0000
5   Tomcat02   log.error_count     31     0.0000
4   Tomcat01     log.row_count     31   612.0000
13  Tomcat04     log.row_count     31   607.0000
11  Tomcat04   log.error_count     31     0.0000
10  Tomcat03     log.row_count     31   616.5000
8   Tomcat03   log.error_count     31     0.0000
15  apache02     log.row_count     31  1206.0000
14  apache02   log.error_count     31     0.0000
1    Mysql01     log.row_count     30    18.0000
0    Mysql01   log.error_count     30     0.0000
6   Tomcat02  log.gc_pause_max     30     0.1000
3   Tomcat01  log.gc_pause_max     26     0.0875
9   Tomcat03  log.gc_pause_max     26     0.0900
12  Tomcat04  log.gc_pause_max     26     0.0800, Empty DataFrame
Columns: [cmdb_id, count, earliest, latest, messages]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_df (loaded previously)
# Define candidate components
candidates = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Ensure metric_df exists and value is numeric, timestamp is timezone-aware Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
# If timestamp not tz-aware, convert; but earlier steps made it tz-aware. Ensure consistency:
if metric_df['timestamp'].dtype == object or str(metric_df['timestamp'].dtype).startswith('datetime64') and metric_df['timestamp'].dt.tz is None:
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 2) Restrict metric rows to candidate set for threshold calculation
metric_candidate = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Compute global P95, P05 and total count for each (cmdb_id, kpi_name) pair using ALL available rows (for candidates)
metric_thresholds = (
    metric_candidate
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        total_count='count',
        global_p95=lambda x: x.quantile(0.95),
        global_p05=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# Round thresholds for compactness
metric_thresholds['global_p95'] = metric_thresholds['global_p95'].round(6)
metric_thresholds['global_p05'] = metric_thresholds['global_p05'].round(6)

# Store metric_thresholds in kernel (already assigned)

# 3) Filter original metric rows to the incident time window (UTC+8): 2021-03-06 15:00 to 15:30
start = pd.Timestamp('2021-03-06 15:00:00', tz=tz)
end   = pd.Timestamp('2021-03-06 15:30:00', tz=tz)

# Filter metric_candidate (so only candidate components are considered in incident)
metric_incident = metric_candidate[(metric_candidate['timestamp'] >= start) & (metric_candidate['timestamp'] <= end)].copy()

# 4) For each (cmdb_id, kpi_name) present in metric_incident detect anomalies using thresholds
if not metric_incident.empty:
    # Merge incident rows with thresholds to get global p95/p05 per row
    incident_with_thresh = metric_incident.merge(
        metric_thresholds,
        on=['cmdb_id','kpi_name'],
        how='left',
        suffixes=('','_th')
    )

    # Determine anomaly flag per row
    incident_with_thresh['anomaly_above'] = incident_with_thresh['value'] >= incident_with_thresh['global_p95']
    incident_with_thresh['anomaly_below'] = incident_with_thresh['value'] <= incident_with_thresh['global_p05']
    incident_with_thresh['is_anomaly'] = incident_with_thresh['anomaly_above'] | incident_with_thresh['anomaly_below']

    # Aggregate per pair
    def compute_pair_agg(df):
        total_incident_points = len(df)
        anomaly_count = int(df['is_anomaly'].sum())
        # earliest/latest anomaly timestamps
        if anomaly_count > 0:
            earliest = df.loc[df['is_anomaly'], 'timestamp'].min()
            latest = df.loc[df['is_anomaly'], 'timestamp'].max()
        else:
            earliest = pd.NaT
            latest = pd.NaT
        max_val = df['value'].max()
        min_val = df['value'].min()
        global_p95 = df['global_p95'].iloc[0]
        global_p05 = df['global_p05'].iloc[0]

        # severity calculation: only use applicable sides
        sev_candidates = []
        # above p95 side
        if (df['anomaly_above'].any()) and pd.notnull(global_p95) and global_p95 != 0:
            sev_candidates.append(abs(max_val / global_p95 - 1))
        # below p05 side
        if (df['anomaly_below'].any()) and pd.notnull(min_val) and min_val != 0:
            sev_candidates.append(abs(global_p05 / min_val - 1))
        severity = float(np.max(sev_candidates)) if sev_candidates else 0.0

        return pd.Series({
            'total_incident_points': total_incident_points,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest,
            'latest_anomaly_timestamp': latest,
            'max_value_in_incident': max_val,
            'min_value_in_incident': min_val,
            'global_p95': global_p95,
            'global_p05': global_p05,
            'severity': severity
        })

    anomalies_summary = (
        incident_with_thresh
        .groupby(['cmdb_id','kpi_name'], dropna=False)
        .apply(compute_pair_agg)
        .reset_index()
    )

    # Format timestamps for compact display
    anomalies_summary['earliest_anomaly_timestamp'] = anomalies_summary['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    anomalies_summary['latest_anomaly_timestamp'] = anomalies_summary['latest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    # Sort by anomaly_count desc then severity desc and take top 20
    anomalies_top20 = anomalies_summary.sort_values(['anomaly_count','severity'], ascending=[False, False]).head(20)

    # Also prepare metric_thresholds rows for those returned pairs
    # Merge to get compact thresholds for returned pairs
    top_pairs = anomalies_top20[['cmdb_id','kpi_name']]
    thresholds_for_top = top_pairs.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')[
        ['cmdb_id','kpi_name','global_p95','global_p05','total_count']
    ]
    # Round numeric columns for compactness
    thresholds_for_top['global_p95'] = thresholds_for_top['global_p95'].round(6)
    thresholds_for_top['global_p05'] = thresholds_for_top['global_p05'].round(6)
else:
    # No incident rows found
    anomalies_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_incident_points','anomaly_count',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp',
        'max_value_in_incident','min_value_in_incident','global_p95','global_p05','severity'
    ])
    anomalies_top20 = anomalies_summary
    thresholds_for_top = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','global_p05','total_count'])

# Store metric_thresholds and metric_incident in kernel (already assigned above)
# Return compact aggregates: anomalies_top20 and thresholds_for_top
anomalies_top20, thresholds_for_top

```
Out[2]:
```
Summary (plain English):

- Candidate set used: apache01, apache02, Tomcat01–04, MG01, MG02, IG01, IG02, Mysql01, Mysql02, Redis01, Redis02.
- Incident window: 2021-03-06 15:00 → 2021-03-06 15:30 (UTC+8).
- Kernel variables stored for follow-up: metric_thresholds (global p95/p05 per candidate pair) and metric_incident (all candidate metric rows in the incident window).

Key findings (top pairs by anomaly count):
- All reported top pairs had anomaly_count = total incident points (31/31), i.e., every sampled minute in the 15:00–15:30 window exceeded the pair’s global threshold (>=P95 or <=P05).
- Highest-severity items:
  1. apache02 — OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_… : anomaly_count 31, severity 0.25 (max value 0.50 vs global P95 0.40 → ~25% above P95).
  2. apache01 — OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_… : anomaly_count 31, severity 0.1667 (~16.7% above P95).
- Many JVM memory and network KPI pairs for IG01, IG02, MG01, MG02, Mysql01, Mysql02 also show anomaly_count 31 but severity 0.0 (these values equal the stored global P95/P05, so they are flagged as at-threshold rather than strongly deviating).

Interpretation and next steps:
- The most actionable anomalies point to apache01/apache02 zabbix-agent related process metrics (notably apache02 with the largest relative exceedance). These should be investigated first.
- Several JVM heap and OS network/process KPIs are at or beyond their historical thresholds across IG/MG/Mysql nodes; verify whether thresholds are correct and inspect those hosts’ logs and traces for correlated issues.
- metric_thresholds and metric_incident are available in the kernel for deeper inspection or to correlate with logs/traces.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  total_incident_points  anomaly_count earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_incident  min_value_in_incident    global_p95    global_p05  severity
1579  apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           5.000000e-01           4.000000e-01  4.000000e-01  4.000000e-01  0.250000
1504  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           7.000000e-01           5.000000e-01  6.000000e-01  5.000000e-01  0.166667
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           2.040136e+09           2.040136e+09  2.040136e+09  2.040136e+09  0.000000
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           5.800194e+07           5.798357e+07  5.800194e+07  5.798357e+07  0.000000
66        IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
67        IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
88        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           2.040136e+09           2.040136e+09  2.040136e+09  2.040136e+09  0.000000
154       IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
155       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           1.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
176       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           2.040136e+09           2.040136e+09  2.040136e+09  2.040136e+09  0.000000
179       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           5.068464e+08           5.068464e+08  5.068464e+08  5.068464e+08  0.000000
268       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           2.040136e+09           2.040136e+09  2.040136e+09  2.040136e+09  0.000000
271       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           9.472855e+08           9.472855e+08  9.472855e+08  9.472855e+08  0.000000
572    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
573    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
575    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
578    Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           4.000000e-01           3.000000e-01  4.000000e-01  3.000000e-01  0.000000
579    Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           7.000000e+00           7.000000e+00  7.000000e+00  7.000000e+00  0.000000
796    Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
797    Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                     31             31    2021-03-06 15:00:00 CST  2021-03-06 15:30:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000,      cmdb_id                                           kpi_name    global_p95    global_p05  total_count
0   apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  4.000000e-01  4.000000e-01           31
1   apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  6.000000e-01  5.000000e-01           31
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09           31
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.800194e+07  5.798357e+07           31
4       IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00           31
5       IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00           31
6       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09           31
7       IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00           31
8       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00           31
9       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09           31
10      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.068464e+08  5.068464e+08           31
11      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09           31
12      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  9.472855e+08  9.472855e+08           31
13   Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00           31
14   Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00           31
15   Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  0.000000e+00  0.000000e+00           31
16   Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  4.000000e-01  3.000000e-01           31
17   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  7.000000e+00  7.000000e+00           31
18   Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00           31
19   Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00           31)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel: trace_df, candidates, tz (Asia/Shanghai)
# Ensure tz and trace_df exist and are correctly typed
tz = pytz.timezone('Asia/Shanghai')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
# Ensure timestamp is timezone-aware Asia/Shanghai
if trace_df['timestamp'].dtype == object or (hasattr(trace_df['timestamp'].dt, 'tz') and trace_df['timestamp'].dt.tz is None):
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P05 for each (cmdb_id, trace_name) using all rows in trace.csv
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95),
         global_p05=lambda x: x.quantile(0.05),
         total_count='count')
    .reset_index()
)
trace_thresholds['global_p95'] = trace_thresholds['global_p95'].round(6)
trace_thresholds['global_p05'] = trace_thresholds['global_p05'].round(6)

# 2) Filter trace.csv to incident window and candidate components
start = pd.Timestamp('2021-03-06 15:00:00', tz=tz)
end   = pd.Timestamp('2021-03-06 15:30:00', tz=tz)
trace_incident = trace_df[
    (trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end) & (trace_df['cmdb_id'].isin(candidates))
].copy()

# 3) Detect anomalies per (cmdb_id, trace_name) in trace_incident
if not trace_incident.empty:
    # Merge with thresholds to attach global p95/p05
    incident_with_thresh = trace_incident.merge(
        trace_thresholds,
        on=['cmdb_id','trace_name'],
        how='left'
    )

    # Define drop-type traces (business/traffic counts or success rates) by name heuristics
    drop_keywords = ['row_count','rowcount','count','cnt','sr','rr','rate','success','throughput','req','request']
    def is_drop_trace(name):
        if not isinstance(name, str):
            return False
        nl = name.lower()
        return any(k in nl for k in drop_keywords)

    incident_with_thresh['is_drop_trace'] = incident_with_thresh['trace_name'].apply(is_drop_trace)

    # Determine anomaly flags: for drop traces anomaly when value <= global_p05, else when value >= global_p95
    incident_with_thresh['anomaly_above'] = incident_with_thresh['value'] >= incident_with_thresh['global_p95']
    incident_with_thresh['anomaly_below'] = incident_with_thresh['value'] <= incident_with_thresh['global_p05']
    incident_with_thresh['is_anomaly'] = np.where(
        incident_with_thresh['is_drop_trace'],
        incident_with_thresh['anomaly_below'],
        incident_with_thresh['anomaly_above']
    )

    # Aggregate per pair
    def agg_trace_pair(df):
        total_incident_points = len(df)
        anomaly_count = int(df['is_anomaly'].sum())
        if anomaly_count > 0:
            earliest = df.loc[df['is_anomaly'], 'timestamp'].min()
            latest = df.loc[df['is_anomaly'], 'timestamp'].max()
        else:
            earliest = pd.NaT
            latest = pd.NaT
        max_val = df['value'].max()
        min_val = df['value'].min()
        global_p95 = df['global_p95'].iloc[0]
        global_p05 = df['global_p05'].iloc[0]
        # severity: for drop traces use drop severity (global_p05 / min - 1), else use (max / global_p95 -1)
        if df['is_drop_trace'].iloc[0]:
            if pd.isna(min_val) or min_val == 0 or pd.isna(global_p05) or global_p05 == 0:
                severity = 0.0
            else:
                severity = float(global_p05 / min_val - 1)
        else:
            if pd.isna(global_p95) or global_p95 == 0:
                severity = 0.0
            else:
                severity = float(max_val / global_p95 - 1)
        return pd.Series({
            'total_incident_points': total_incident_points,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest,
            'latest_anomaly_timestamp': latest,
            'max_value_in_incident': max_val,
            'global_p95': global_p95,
            'severity': severity
        })

    trace_anomalies = (
        incident_with_thresh
        .groupby(['cmdb_id','trace_name'], dropna=False)
        .apply(agg_trace_pair)
        .reset_index()
    )

    # Format timestamps compactly
    trace_anomalies['earliest_anomaly_timestamp'] = trace_anomalies['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    trace_anomalies['latest_anomaly_timestamp'] = trace_anomalies['latest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    # Sort by anomaly_count desc then severity desc and take top 20
    trace_top20 = trace_anomalies.sort_values(['anomaly_count','severity'], ascending=[False, False]).head(20)

    # For compactness ensure rounding of numeric columns
    trace_top20['max_value_in_incident'] = trace_top20['max_value_in_incident'].round(6)
    trace_top20['global_p95'] = trace_top20['global_p95'].round(6)
    trace_top20['severity'] = trace_top20['severity'].round(6)

else:
    trace_top20 = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_incident_points','anomaly_count',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp',
        'max_value_in_incident','global_p95','severity'
    ])

# Store trace_thresholds and trace_incident in kernel (variables already assigned)
# Return compact aggregates: trace_top20 and a small slice of trace_thresholds for those pairs
if not trace_top20.empty:
    top_pairs = trace_top20[['cmdb_id','trace_name']].merge(
        trace_thresholds, on=['cmdb_id','trace_name'], how='left'
    )[['cmdb_id','trace_name','global_p95','global_p05','total_count']]
    top_pairs['global_p95'] = top_pairs['global_p95'].round(6)
    top_pairs['global_p05'] = top_pairs['global_p05'].round(6)
else:
    top_pairs = pd.DataFrame(columns=['cmdb_id','trace_name','global_p95','global_p05','total_count'])

# Display compact results
trace_top20, top_pairs

```
Out[3]:
```
Summary (plain English):

- Context: analysis used the candidate components list and the incident window 2021-03-06 15:00 → 15:30 (UTC+8). Full tables trace_thresholds and trace_incident are stored in the kernel for follow-up.

Key findings (top anomalous trace pairs, compact):

1) MG02 — trace.to_dockerA2.row_count
   - Incident points: 30, anomaly_count: 3 (15:07 → 15:27)
   - max in incident: 334, global P95: 231, severity ≈ 0.1417
   - Interpretation: intermittent traffic spikes above typical 95th percentile.

2) IG01 — trace.to_Tomcat02.row_count
   - Incident points: 30, anomaly_count: 3 (15:00 → 15:14)
   - max: 310, global P95: 295.55, severity ≈ 0.0443
   - Interpretation: small traffic increase to Tomcat02 from IG01.

3) Tomcat02 — trace.from_IG01.row_count
   - Same counts/times as IG01 above (symmetrical view).

4) MG01 — trace.to_dockerA2.row_count
   - Incident points: 30, anomaly_count: 3 (15:02 → 15:27)
   - max: 345, global P95: 229.15, severity ≈ 0.0348
   - Interpretation: another traffic spike toward dockerA2 from MG01.

5) MG02 — trace.from_dockerB1.duration_mean
   - Incident points: 30, anomaly_count: 2 (15:09 → 15:29)
   - max: 0.34879, global P95: 0.126051, severity ≈ 1.7671
   - Interpretation: significant latency increase on calls from dockerB1 to MG02.

Other notable patterns:
- Multiple MG01/MG02 traces show anomalies (row_count and duration metrics). Many row_count entries show 2–3 anomalous minutes (moderate increases vs baseline).
- Several duration-related traces (trace.from_... .duration_mean or .duration_p95) on MG nodes have high severity (e.g., severity > 0.9 up to ~1.77), indicating notable latency spikes for specific call paths.
- Trace anomalies often involve MG ↔ docker/Tomcat interactions (e.g., to/from dockerA/B, to/from IG01/Tomcat02).

Recommended next steps:
- Prioritize investigating MG02 and MG01 latency anomalies (duration_mean / duration_p95), because their severity is highest and they indicate real latency increases.
- Check the traffic spikes (to dockerA2 and between IG01↔Tomcat02) for load bursts or routing issues—these could correlate to the latency spikes.
- Correlate these trace anomalies with the stored metric_incident and metric_thresholds (kernel variables) and with logs to determine root cause (resource exhaustion, network issues, or application slowness).
- Use the stored trace_thresholds and trace_incident for drilling into the exact timestamps and full trace samples.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                         trace_name  total_incident_points  anomaly_count earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_incident  global_p95  severity
113      MG02        trace.to_dockerA2.row_count                     30              3    2021-03-06 15:07:00 CST  2021-03-06 15:27:00 CST             334.000000  231.000000  0.141732
11       IG01        trace.to_Tomcat02.row_count                     30              3    2021-03-06 15:00:00 CST  2021-03-06 15:14:00 CST             310.000000  295.550000  0.044335
140  Tomcat02          trace.from_IG01.row_count                     30              3    2021-03-06 15:00:00 CST  2021-03-06 15:14:00 CST             310.000000  295.550000  0.044335
71       MG01        trace.to_dockerA2.row_count                     30              3    2021-03-06 15:02:00 CST  2021-03-06 15:27:00 CST             345.000000  229.150000  0.034783
96       MG02  trace.from_dockerB1.duration_mean                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:29:00 CST               0.348790    0.126051  1.767055
54       MG01  trace.from_dockerB1.duration_mean                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:20:00 CST               0.312039    0.127758  1.442422
92       MG02      trace.from_dockerA1.row_count                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:15:00 CST             100.000000   84.400000  0.990000
57       MG01  trace.from_dockerB2.duration_mean                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:28:00 CST               0.236183    0.121597  0.942342
98       MG02      trace.from_dockerB1.row_count                     30              2    2021-03-06 15:13:00 CST  2021-03-06 15:19:00 CST             334.000000  292.050000  0.936170
99       MG02  trace.from_dockerB2.duration_mean                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:29:00 CST               0.256378    0.135183  0.896525
53       MG01      trace.from_dockerA2.row_count                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:10:00 CST             112.000000   74.400000  0.845000
93       MG02  trace.from_dockerA2.duration_mean                     30              2    2021-03-06 15:05:00 CST  2021-03-06 15:09:00 CST               0.295857    0.173089  0.709277
97       MG02   trace.from_dockerB1.duration_p95                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:29:00 CST               1.660600    1.018292  0.630770
55       MG01   trace.from_dockerB1.duration_p95                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:20:00 CST               1.501000    1.026188  0.462695
95       MG02      trace.from_dockerA2.row_count                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:12:00 CST             102.000000   93.800000  0.382143
51       MG01  trace.from_dockerA2.duration_mean                     30              2    2021-03-06 15:09:00 CST  2021-03-06 15:21:00 CST               0.208000    0.157075  0.324208
59       MG01      trace.from_dockerB2.row_count                     30              2    2021-03-06 15:13:00 CST  2021-03-06 15:19:00 CST             317.000000  304.150000  0.307143
56       MG01      trace.from_dockerB1.row_count                     30              2    2021-03-06 15:13:00 CST  2021-03-06 15:19:00 CST             286.000000  273.400000  0.302885
39       MG01  trace.from_Tomcat02.duration_mean                     30              2    2021-03-06 15:16:00 CST  2021-03-06 15:29:00 CST               0.944835    0.731836  0.291047
150  Tomcat02        trace.to_MG01.duration_mean                     30              2    2021-03-06 15:16:00 CST  2021-03-06 15:29:00 CST               0.944835    0.731836  0.291047,      cmdb_id                         trace_name  global_p95  global_p05  total_count
0       MG02        trace.to_dockerA2.row_count  231.000000  145.000000           30
1       IG01        trace.to_Tomcat02.row_count  295.550000  212.000000           30
2   Tomcat02          trace.from_IG01.row_count  295.550000  212.000000           30
3       MG01        trace.to_dockerA2.row_count  229.150000  119.000000           30
4       MG02  trace.from_dockerB1.duration_mean    0.126051    0.019747           30
5       MG01  trace.from_dockerB1.duration_mean    0.127758    0.045216           30
6       MG02      trace.from_dockerA1.row_count   84.400000   19.900000           30
7       MG01  trace.from_dockerB2.duration_mean    0.121597    0.037509           30
8       MG02      trace.from_dockerB1.row_count  292.050000   91.000000           30
9       MG02  trace.from_dockerB2.duration_mean    0.135183    0.035176           30
10      MG01      trace.from_dockerA2.row_count   74.400000   18.450000           30
11      MG02  trace.from_dockerA2.duration_mean    0.173089    0.042681           30
12      MG02   trace.from_dockerB1.duration_p95    1.018292    0.011980           30
13      MG01   trace.from_dockerB1.duration_p95    1.026188    0.057435           30
14      MG02      trace.from_dockerA2.row_count   93.800000   19.350000           30
15      MG01  trace.from_dockerA2.duration_mean    0.157075    0.025809           30
16      MG01      trace.from_dockerB2.row_count  304.150000   91.500000           30
17      MG01      trace.from_dockerB1.row_count  273.400000   67.750000           30
18      MG01  trace.from_Tomcat02.duration_mean    0.731836    0.318108           30
19  Tomcat02        trace.to_MG01.duration_mean    0.731836    0.318108           30)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables: metric_incident, metric_thresholds
tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist
# If metric_incident or metric_thresholds don't exist or are empty, create empty outputs
if 'metric_incident' not in globals() or 'metric_thresholds' not in globals():
    top20_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_incident_points','anomaly_count',
        'longest_run_length','longest_run_start',
        'max_value_in_incident','min_value_in_incident',
        'global_p95','global_p05','severity'
    ])
    faults_table = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','run_id','run_start','run_end','run_length',
        'run_count','run_max_value','run_min_value'
    ])
else:
    # Merge incident rows with thresholds to attach global p95/p05
    incident_with_thresh = metric_incident.merge(
        metric_thresholds,
        on=['cmdb_id','kpi_name'],
        how='left'
    ).copy()

    # Ensure timestamp is datetime tz-aware
    if incident_with_thresh['timestamp'].dtype == object or (hasattr(incident_with_thresh['timestamp'].dt, 'tz') and incident_with_thresh['timestamp'].dt.tz is None):
        incident_with_thresh['timestamp'] = pd.to_datetime(incident_with_thresh['timestamp']).dt.tz_convert(tz)

    # Numeric value
    incident_with_thresh['value'] = pd.to_numeric(incident_with_thresh['value'], errors='coerce')

    # Anomaly flags
    incident_with_thresh['anomaly_above'] = incident_with_thresh['value'] >= incident_with_thresh['global_p95']
    incident_with_thresh['anomaly_below'] = incident_with_thresh['value'] <= incident_with_thresh['global_p05']
    incident_with_thresh['is_anomaly'] = incident_with_thresh['anomaly_above'] | incident_with_thresh['anomaly_below']

    # Prepare list to collect per-pair run records
    runs_records = []

    # Aggregate per pair and detect consecutive-minute runs among anomaly points
    group_cols = ['cmdb_id','kpi_name']
    agg_rows = []
    for (cmdb, kpi), grp in incident_with_thresh.groupby(group_cols):
        grp_sorted = grp.sort_values('timestamp').reset_index(drop=True)
        total_incident_points = len(grp_sorted)
        anomaly_count = int(grp_sorted['is_anomaly'].sum())
        max_val = grp_sorted['value'].max() if total_incident_points>0 else np.nan
        min_val = grp_sorted['value'].min() if total_incident_points>0 else np.nan
        global_p95 = grp_sorted['global_p95'].iloc[0] if 'global_p95' in grp_sorted.columns else np.nan
        global_p05 = grp_sorted['global_p05'].iloc[0] if 'global_p05' in grp_sorted.columns else np.nan

        # Severity: consider applicable sides only when anomalies exist above p95 or below p05
        sev_candidates = []
        if grp_sorted['anomaly_above'].any() and pd.notnull(global_p95) and global_p95 != 0:
            sev_candidates.append(abs(max_val / global_p95 - 1))
        if grp_sorted['anomaly_below'].any() and pd.notnull(min_val) and min_val != 0:
            sev_candidates.append(abs(global_p05 / min_val - 1))
        severity = float(np.max(sev_candidates)) if sev_candidates else 0.0

        # Detect runs among anomaly points
        anomaly_rows = grp_sorted[grp_sorted['is_anomaly']].copy()
        if not anomaly_rows.empty:
            anomaly_rows = anomaly_rows.sort_values('timestamp').reset_index(drop=True)
            # compute time diff in minutes from previous anomaly row
            anomaly_rows['time_diff'] = anomaly_rows['timestamp'].diff().fillna(pd.Timedelta(seconds=0))
            # start new run when time_diff != 1 minute
            anomaly_rows['new_run'] = anomaly_rows['time_diff'] != pd.Timedelta(minutes=1)
            anomaly_rows['run_id'] = anomaly_rows['new_run'].cumsum()  # run id within this pair
            # For each run compute run metrics
            for run_id, run_grp in anomaly_rows.groupby('run_id'):
                run_start = run_grp['timestamp'].min()
                run_end = run_grp['timestamp'].max()
                run_count = len(run_grp)
                run_length = run_count  # minutes, since points are per-minute
                run_max = run_grp['value'].max()
                run_min = run_grp['value'].min()
                runs_records.append({
                    'cmdb_id': cmdb,
                    'kpi_name': kpi,
                    'run_id': f"{cmdb}||{kpi}||{run_id}",
                    'run_start': run_start,
                    'run_end': run_end,
                    'run_length': run_length,
                    'run_count': run_count,
                    'run_max_value': run_max,
                    'run_min_value': run_min
                })
            # determine longest run for this pair
            runs_for_pair = [r for r in runs_records if r['cmdb_id']==cmdb and r['kpi_name']==kpi]
            # find longest by run_length, tie-breaker earliest start
            runs_df_pair = pd.DataFrame(runs_for_pair)
            longest_idx = runs_df_pair['run_length'].idxmax()
            longest_run_length = int(runs_df_pair.loc[longest_idx, 'run_length'])
            longest_run_start = runs_df_pair.loc[longest_idx, 'run_start']
        else:
            longest_run_length = 0
            longest_run_start = pd.NaT

        agg_rows.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'total_incident_points': total_incident_points,
            'anomaly_count': anomaly_count,
            'longest_run_length': longest_run_length,
            'longest_run_start': longest_run_start,
            'max_value_in_incident': max_val,
            'min_value_in_incident': min_val,
            'global_p95': global_p95,
            'global_p05': global_p05,
            'severity': severity
        })

    # Create DataFrames
    faults_table = pd.DataFrame(runs_records)
    # Ensure timestamp tz for run_start/run_end
    if not faults_table.empty:
        faults_table['run_start'] = pd.to_datetime(faults_table['run_start']).dt.tz_convert(tz)
        faults_table['run_end'] = pd.to_datetime(faults_table['run_end']).dt.tz_convert(tz)

    summary_df = pd.DataFrame(agg_rows)
    # Format longest_run_start for compact display
    if not summary_df.empty:
        summary_df['longest_run_start'] = pd.to_datetime(summary_df['longest_run_start']).dt.tz_convert(tz)
        summary_df['longest_run_start_fmt'] = summary_df['longest_run_start'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        summary_df['longest_run_start_fmt'] = []

    # Select and sort top 20 pairs by longest_run_length desc, then severity desc
    top20_summary = summary_df.sort_values(['longest_run_length','severity'], ascending=[False, False]).head(20)

    # Keep only requested columns and format numerics/timestamps compactly
    if not top20_summary.empty:
        top20_summary = top20_summary[[
            'cmdb_id','kpi_name','total_incident_points','anomaly_count',
            'longest_run_length','longest_run_start_fmt',
            'max_value_in_incident','min_value_in_incident',
            'global_p95','global_p05','severity'
        ]].rename(columns={'longest_run_start_fmt':'longest_run_start'})
        # round numeric columns
        for c in ['max_value_in_incident','min_value_in_incident','global_p95','global_p05','severity']:
            top20_summary[c] = pd.to_numeric(top20_summary[c], errors='coerce').round(6)
    else:
        top20_summary = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','total_incident_points','anomaly_count',
            'longest_run_length','longest_run_start',
            'max_value_in_incident','min_value_in_incident',
            'global_p95','global_p05','severity'
        ])

    # For compact display of faults_table, prepare a head
    faults_table_head = faults_table.sort_values(['run_length'], ascending=False).head(20)

# Final compact output: top20_summary and a compact head of faults_table for follow-up
# (Both variables stored in kernel: top20_summary, faults_table)
top20_summary, faults_table_head

```
Out[4]:
```
Summary (plain English):

- Objective: detect consecutive-minute anomaly runs per (cmdb_id, kpi_name) in the incident window (2021-03-06 15:00 → 15:30 UTC+8). Full per-pair runs table (faults_table) and the per-pair summary (top20_summary) are stored in the kernel for follow-up.

Key findings (compact):

1. Persistent full-window anomalies (longest_run_length = 31 minutes)
   - Multiple KPIs had continuous anomalies for the entire 31-minute incident window (anomaly_count = 31, longest_run_start = 2021-03-06 15:00:00):
     - apache02 — OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_… (max 0.5, global_p95 0.4, severity 0.25)
     - apache01 — OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_… (max 0.7, global_p95 0.6, severity 0.166667)
     - many IG/MG/Mysql/Tomcat/Redis process, memory, network KPIs (JVM heap maxima, process counts, TCP states, etc.) also show continuous anomaly runs for the full window. For many of these the severity is 0.0 because the incident values equal the stored global thresholds (i.e., at or exactly at the saved P95/P05).

2. Notable high-severity continuous items
   - apache02 zabbix-agent process metric is the highest non-zero severity among full-window runs (~25% above P95).
   - apache01 zabbix-agent process metric is next (~16.7% above P95).
   - Several other KPIs are continuously anomalous but with severity 0.0 (values equal thresholds), which still indicates persistent at-threshold behavior.

3. Run-level examples (from faults_table head)
   - Many runs span 2021-03-06 15:00 → 2021-03-06 15:30 (run_length 31).
   - Example run entries include process/no-zombies, Tomcat request/error/max-time metrics, JVM heap max values, and network TCP CLOSE/FIN wait counts — all showing full-window runs.

Interpretation and recommended next steps:

- The most actionable signals are the apache01/apache02 zabbix-agent process metrics (continuous and notably above P95). Investigate those processes on apache hosts first (why the agent/process metric is elevated).
- The widespread full-window anomalies across many host/KPI pairs suggest either:
  - a real, system-wide shift during the incident, or
  - threshold values that coincide with typical operating points (leading to many severity=0 flags). Validate the correctness of historical thresholds for those KPIs.
- Use the stored faults_table and top20_summary in the kernel to:
  - drill into exact timestamps and run-level records,
  - correlate these continuous anomalies with trace anomalies and logs (trace_incident, metric_incident, log data),
  - inspect host resource usage and application logs around 15:00–15:30 for root-cause (e.g., zabbix agent behavior, JVM heap events, network state).

Stored kernel artifacts for follow-up:
- top20_summary (per-pair summary with longest run metrics)
- faults_table (full per-run records)

These contain the detailed run start/end times and per-run min/max values for deeper analysis.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  total_incident_points  anomaly_count  longest_run_length        longest_run_start  max_value_in_incident  min_value_in_incident    global_p95    global_p05  severity
1579  apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                     31             31                  31  2021-03-06 15:00:00 CST           5.000000e-01           4.000000e-01  4.000000e-01  4.000000e-01  0.250000
1504  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                     31             31                  31  2021-03-06 15:00:00 CST           7.000000e-01           5.000000e-01  6.000000e-01  5.000000e-01  0.166667
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                     31             31                  31  2021-03-06 15:00:00 CST           2.040136e+09           2.040136e+09  2.040136e+09  2.040136e+09  0.000000
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                     31             31                  31  2021-03-06 15:00:00 CST           5.800194e+07           5.798357e+07  5.800194e+07  5.798357e+07  0.000000
66        IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                     31             31                  31  2021-03-06 15:00:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
67        IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                     31             31                  31  2021-03-06 15:00:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
88        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                     31             31                  31  2021-03-06 15:00:00 CST           2.040136e+09           2.040136e+09  2.040136e+09  2.040136e+09  0.000000
154       IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                     31             31                  31  2021-03-06 15:00:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
155       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                     31             31                  31  2021-03-06 15:00:00 CST           1.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
176       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                     31             31                  31  2021-03-06 15:00:00 CST           2.040136e+09           2.040136e+09  2.040136e+09  2.040136e+09  0.000000
179       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                     31             31                  31  2021-03-06 15:00:00 CST           5.068464e+08           5.068464e+08  5.068464e+08  5.068464e+08  0.000000
268       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                     31             31                  31  2021-03-06 15:00:00 CST           2.040136e+09           2.040136e+09  2.040136e+09  2.040136e+09  0.000000
271       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                     31             31                  31  2021-03-06 15:00:00 CST           9.472855e+08           9.472855e+08  9.472855e+08  9.472855e+08  0.000000
572    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                     31             31                  31  2021-03-06 15:00:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
573    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                     31             31                  31  2021-03-06 15:00:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
575    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                     31             31                  31  2021-03-06 15:00:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
578    Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                     31             31                  31  2021-03-06 15:00:00 CST           4.000000e-01           3.000000e-01  4.000000e-01  3.000000e-01  0.000000
579    Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                     31             31                  31  2021-03-06 15:00:00 CST           7.000000e+00           7.000000e+00  7.000000e+00  7.000000e+00  0.000000
796    Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                     31             31                  31  2021-03-06 15:00:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000
797    Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                     31             31                  31  2021-03-06 15:00:00 CST           0.000000e+00           0.000000e+00  0.000000e+00  0.000000e+00  0.000000,         cmdb_id                                           kpi_name                                             run_id                 run_start                   run_end  run_length  run_count  run_max_value  run_min_value
10238  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  apache02||OSLinux-OSLinux_PROCESS_PROCESS_PROC... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00
7783   Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter  Tomcat01||Tomcat-Sessions_7441--UOCP_SESSIONAc... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00
608        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  IG02||JVM-Memory_7778_JVM_Memory_HeapMemoryMax||1 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   2.040136e+09   2.040136e+09
7778   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  Tomcat01||Tomcat-Requests_7441-"http-nio-8003"... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   3.488470e+05   3.488470e+05
7777   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  Tomcat01||Tomcat-Requests_7441-"http-nio-8003"... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   2.300000e+01   2.300000e+01
5754    Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  Mysql02||OSLinux-OSLinux_PROCESS_zabbix-zabbix... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   5.000000e-01   5.000000e-01
5755    Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  Mysql02||OSLinux-OSLinux_PROCESS_zabbix_PROCPP... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   1.200000e+01   1.200000e+01
6364    Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  Redis01||OSLinux-OSLinux_NETWORK_NETWORK_TCP-F... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   3.000000e+00   0.000000e+00
6363    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  Redis01||OSLinux-OSLinux_NETWORK_NETWORK_TCP-C... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00
9616   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  Tomcat04||Tomcat-Threads_7441-"http-nio-8003"_... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   5.000000e+02   5.000000e+02
1064       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  IG02||OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   1.000000e+00   0.000000e+00
9610   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  Tomcat04||Tomcat-Threads_7441-"http-nio-8003"_... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   1.000000e+02   1.000000e+02
8384   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  Tomcat02||Tomcat-Requests_7441-"http-nio-8003"... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   3.991480e+05   3.991480e+05
9608   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  Tomcat04||Tomcat-Sessions_7441--logHome_IS_UND... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00
9607   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  Tomcat04||Tomcat-Sessions_7441--logHome_IS_UND... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00
9606   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  Tomcat04||Tomcat-Sessions_7441--log.Home_IS_UN... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00
9605   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  Tomcat04||Tomcat-Sessions_7441--log.Home_IS_UN... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00
9604   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  Tomcat04||Tomcat-Sessions_7441--log.Home_IS_UN... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00
9603   Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions  Tomcat04||Tomcat-Sessions_7441--_SESSIONReject... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00
9602   Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter  Tomcat04||Tomcat-Sessions_7441--_SESSIONKeepal... 2021-03-06 15:00:00+08:00 2021-03-06 15:30:00+08:00          31         31   0.000000e+00   0.000000e+00)```
```